# M2 · Evaluation harness — MacGyver Gym Rat

**SI4006 · Universidad EAFIT · Module 2** · runs end to end on a free Colab T4.

This notebook measures the M1 system on three dimensions and writes
`reports/scorecard_baseline.csv`. It does not train anything.

| Dimension | What it asks | Where it lives |
|---|---|---|
| 1 · classic metric | Does the answer *look* like a valid one? Embedding cosine + ROUGE-L of the recited steps. | `eval/harness.py` |
| 2 · LLM-as-a-judge | Is it correct, safe and appropriate? 1-5 against a versioned rubric. | `eval/rubric.md` |
| 3 · domain hit rate | Does the catalog agree the exercise is real, for that muscle, with that object — and does the system refuse when it should? | `eval/harness.py` |

Two systems get the same yardstick: the base model with no adapter (zero-shot)
and the M1 LoRA. In M3 the retrieval system becomes a third column, and nothing
in this notebook changes except the callable passed to `harness()`.

**Runtime → Change runtime type → T4 GPU.** On CPU this finishes, but slowly.


## 0 · Setup

Clone the repo *with the dataset submodule* — the catalog is what grades dimension 3, and without it nothing below runs.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/iamcroody/models-for-exercises-dataset.git"
# eval/ and this notebook live on main since PR #4 was merged, so this clones
# the default branch. Pointing at a feature branch that has already been merged
# would break the day someone tidies it up.
BRANCH = "main"
ROOT = Path("/content/models-for-exercises-dataset")

if "google.colab" in sys.modules:
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--recurse-submodules", "-b", BRANCH,
                        REPO, str(ROOT)], check=True)
    os.chdir(ROOT)
else:
    # Running from the repo itself (local, or `jupyter` in the project).
    ROOT = Path.cwd() if (Path.cwd() / "eval").exists() else Path.cwd().parent
    os.chdir(ROOT)

print("working in", Path.cwd())
if not (Path.cwd() / "eval/harness.py").exists():
    raise SystemExit("eval/ is missing: clone the repo at a commit that "
                     "carries it (PR #4 or later)")
if not (Path.cwd() / "data/exercises-dataset/data/exercises.json").exists():
    raise SystemExit("the dataset submodule is missing: "
                     "git submodule update --init --recursive")

In [ ]:
# Colab ships torch and transformers, but both `mg.load_model` and the judge pass
# `dtype=` to `from_pretrained`, which the preinstalled transformers is too old to
# accept. So transformers is upgraded rather than trusted — the same recipe M1 uses,
# including the torchao removal, so both modules run on the same stack.
%pip install -q -U transformers peft
%pip install -q sentence-transformers rouge_score
# Colab pins torchao 0.10 to its torch build; peft refuses anything below 0.16,
# and upgrading it drags a different torch in behind it.
%pip uninstall -y -q torchao

import transformers
print("transformers", transformers.__version__)
print("done — if Colab asks you to restart the runtime, restart and re-run from cell 1")

In [ ]:
import json, gc, torch
sys.path.insert(0, "scripts")
sys.path.insert(0, "eval")

import macgyver as mg
import harness as H

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__)
print("rubric:", H.load_rubric()["version"], "| seed:", H.SEED)

## 1 · The eval set

Twelve examples, four of them adversarial (33%). Built by `eval/build_eval_set.py`
from the catalog, so every reference answer's steps are the real ones; the
(muscle, object) pairs were chosen by hand and none of them appears in the M1
training split.

Run `python eval/build_eval_set.py` to rebuild it — the script refuses to write a
contaminated or unanswerable example.

In [ ]:
eval_set = H.load_eval_set()

print(f"{len(eval_set)} examples, "
      f"{sum(e['adversarial'] for e in eval_set)} adversarial\n")
for e in eval_set:
    meta = e["meta"]
    tag = "ADV" if e["adversarial"] else "   "
    obj = (meta.get("object") or "-")[:34]
    print(f"{tag} {e['id']:26} {e['kind']:14} {str(meta.get('target')):12} {obj}")

In [ ]:
# Sanity check with no model weights: a system that replies with the reference
# answer must score 1.0 on dimension 3, and one that always says "plank" must
# score 0.0. If this fails, every number below is meaningless.
H.self_test();

## 2 · Generating the replies

The free T4 will not hold the generator, the judge and the embedding model at
once, so we generate first, free the GPU, and judge afterwards. The harness sees
a callable either way (`H.replay`), so the measurement is unaffected.

Greedy decoding, `enable_thinking=False`, the same 448-token budget M1 used.

In [ ]:
def generate(adapter=None, label=""):
    tok = mg.load_tokenizer()
    model = mg.load_model(mg.BASE_MODEL, adapter=adapter)
    system = H.make_hf_system(model, tok)

    replies = {}
    for i, e in enumerate(eval_set, 1):
        replies[e["input"]] = system(e["input"])
        print(f"  {label} {i}/{len(eval_set)}", end="\r", flush=True)
    print()

    # `system` closes over model and tok, so deleting only those two names frees
    # nothing and empty_cache runs with the weights still referenced — which is
    # how the second generate() call OOMs on a free T4.
    del system, model, tok
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return replies

replies_zeroshot = generate(adapter=None, label="zero-shot")
replies_lora = generate(adapter="models/r16-all-linear", label="lora")

Path("reports").mkdir(exist_ok=True)
Path("reports/replies_m2.json").write_text(json.dumps(
    {"zeroshot": replies_zeroshot, "lora": replies_lora}, indent=2, ensure_ascii=False))
print("replies cached in reports/replies_m2.json")

In [ ]:
# One example, to see what we are actually grading.
sample = eval_set[4]
print(sample["input"][:180], "...\n")
print("--- LoRA reply ---")
print(replies_lora[sample["input"]][:700])

## 3 · Dimension 1 — the classic metric

Cosine similarity between sentence embeddings, plus ROUGE-L of the recited steps
against the real steps of the exercise the model *named*.

Why both: similarity alone rewards an answer that reads like the reference, and
in this domain "reads like it" and "is safe to do" are different questions. ROUGE
is scored against the named exercise rather than the reference exercise, so
picking a different but valid exercise is not punished.

In [ ]:
similarity = H.Similarity()

print("paraphrase :", round(similarity("The cat sleeps.", "The feline is resting."), 2))
print("unrelated  :", round(similarity("The cat sleeps.", "The car is red."), 2))

## 4 · Dimension 2 — the judge

`Qwen/Qwen2.5-1.5B-Instruct`, small enough for the free tier, scoring 1-5 against
`eval/rubric.md` (rubric **v1.0**). The rubric text is read from that file at
runtime, so a score can always be traced to the exact wording that produced it.

The parser returns `None` rather than a neutral 3 when the judge fails to emit a
digit: a judge that quietly scores 3 whenever it rambles looks like an average
judge instead of a broken one.

In [ ]:
judge = H.Judge(device=DEVICE)
print("judge:", judge.model_id, "| rubric:", judge.rubric["version"])
print()
print(judge.rubric["pointwise"][:600], "...")

In [ ]:
# Does the judge separate a good answer from a bad one at all? If it does not,
# dimension 2 is noise and should be reported as such.
case = eval_set[4]
good = case["expected"]
poor = ("Exercise: Mega Brick Curl 3000\nGym equivalent: barbell\n"
        "Adaptation: Tie the bricks to your wrists with a belt and swing hard.\n"
        "Steps:\n1. Swing as fast as you can for two minutes.\n"
        "Safety: No need to warm up.")

print("good answer ->", judge.score(case["input"], good, case["expected"]), "/ 5")
print("poor answer ->", judge.score(case["input"], poor, case["expected"]), "/ 5")

## 5 · Judge bias — measured, then mitigated

Two known biases, both probed on our own data rather than cited from a paper.

**Position bias.** Ask the judge to pick the better of two answers, then ask
again with the answers swapped. Every disagreement is a verdict that came from
the seating order, not the content. Mitigation: `judge.compare_robust` asks both
ways and only declares a winner when the two agree — otherwise it is a tie.

**Length / verbosity bias.** Score each reply, then score it again padded with
content-free filler. Any positive mean delta is the judge paying for words.
Mitigation: the rubric says in as many words that length is not quality, and the
judge never sees more than `MAX_ANSWER_CHARS` of an answer, so padding cannot
buy a score with text that is not shown.

In [ ]:
gold_replies = [replies_lora[e["input"]] for e in eval_set]

print("position bias, reference answer vs LoRA reply, both orders:")
position = H.position_bias_probe(judge, eval_set, gold_replies)
print(f"\nflip rate: {position['flip_rate']:.0%} "
      f"({position['flips']} of {position['decided']} decided verdicts "
      f"depended on the order)")
print(f"unparsed: {position['unparsed']} of {position['n']} pairs, kept out of "
      f"the rate so 'gave no letter' is not counted as 'changed its mind'")


In [ ]:
print("length bias, same reply padded with filler:")
length = H.length_bias_probe(judge, eval_set, gold_replies)
print(f"\nmean delta, cap lifted : {length['mean_delta_uncapped']} "
      f"({length['raised_uncapped']} raised, {length['lowered_uncapped']} lowered)")
print(f"mean delta, cap applied: {length['mean_delta_capped']} "
      f"({length['raised_capped']} raised, {length['lowered_capped']} lowered)")
print(f"\nthe first number is the bias, the second is what survives the "
      f"{length['max_answer_chars']}-character cap.")


In [ ]:
# The mitigation in use: a robust pairwise verdict on the same pairs.
verdicts = {"x": 0, "y": 0, "tie": 0}
for e, reply in zip(eval_set, gold_replies):
    verdicts[judge.compare_robust(e["input"], e["expected"], reply)] += 1
print("reference wins:", verdicts["x"], "| LoRA wins:", verdicts["y"],
      "| undecided after both orders:", verdicts["tie"])

## 6 · The scorecard

`harness(eval_set, system)` runs all three dimensions over one callable. Two
callables here; in M3 the RAG pipeline is a third.

In [ ]:
print("zero-shot")
sc_zeroshot = H.harness(eval_set, H.replay(replies_zeroshot), judge=judge,
                        similarity=similarity, label="zeroshot")
print("\nLoRA (M1)")
sc_lora = H.harness(eval_set, H.replay(replies_lora), judge=judge,
                    similarity=similarity, label="lora_m1")

H.print_scorecard([sc_zeroshot, sc_lora])
print("judge outputs that did not parse:", judge.unparsed)

In [ ]:
path = H.write_scorecard([sc_zeroshot, sc_lora], "reports/scorecard_baseline.csv")
Path("reports/scorecard_baseline_detail.json").write_text(json.dumps(
    {"zeroshot": sc_zeroshot, "lora_m1": sc_lora,
     "position_bias": position, "length_bias": length},
    indent=2, ensure_ascii=False))
print("wrote", path, "and reports/scorecard_baseline_detail.json")
print(path.read_text())

## 7 · Where it fails

The list below is what goes into the README's honest reading. Read it before
writing that paragraph, not after.

In [ ]:
for name, sc in (("zero-shot", sc_zeroshot), ("LoRA", sc_lora)):
    print(f"\n=== {name}: {sc['domain_hits']}/{sc['n']} domain hits ===")
    for row in sc["detail"]:
        if not row["hit"]:
            print(f"\n  {row['id']} ({row['kind']}) — {row['why_missed']}")
            print(f"    judge={row['judge']} sim={row['similarity']}")
            print("    " + " ".join(row["reply"].split())[:220])

## Reusing this in M3

```python
def rag_system(prompt_text):
    docs = retriever.search(prompt_text)
    return generator(prompt_text, context=docs)

sc_rag = H.harness(eval_set, rag_system, judge=judge, similarity=similarity, label="rag_m3")
H.write_scorecard([sc_zeroshot, sc_lora, sc_rag], "reports/scorecard_m3.csv")
```

Same eval set, same rubric version, same seed. If the rubric changes, bump the
version in `eval/rubric.md` and re-run every column — scores from two rubric
versions do not belong in one table.
